# Compresr × LangChain

You are a competitive-intelligence analyst at a hedge fund. Your LangChain agent reads long company profiles to answer specific questions about business segments, growth, and risk. The problem: every tool call dumps tens of thousands of tokens of mostly-irrelevant content into the model context.

Three groups of integrations fix that. Each section shows agent state **without** and **with** compression on the same realistic analyst workload.

| Group | Symbol | What it does |
|---|---|---|
| **A. Tool wrapper** | `wrap_tool_with_compression` | Wraps any `StructuredTool` so its output is compressed before returning. |
| **B. Middleware** | `CompresrToolMiddleware` | Compresses every tool output as it enters agent state. |
|   | `CompresrSummarizationMiddleware` | Rolls old history into one summary message; KV-cache friendly. |
|   | `CompresrPromptMiddleware` | Last-mile prompt cap right before the model call. |
| **C. Retriever** | `CompresrExtractor` | `BaseDocumentCompressor` — drop-in for `LLMChainExtractor` in RAG. |

Pick the group that matches how you build with LangChain. They compose cleanly.

## Install

In [1]:
%pip install -q -e "..[langchain]" python-dotenv requests openai ipython


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / '.env').exists():
        load_dotenv(parent / '.env')
        break

assert os.environ.get('COMPRESR_API_KEY'), 'COMPRESR_API_KEY not set'
assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
print('API keys loaded.')

API keys loaded.


## Shared setup — a real noisy corpus

Every demo below feeds on a **~55k-token stitched corpus** built from five real Wikipedia articles (Microsoft + Microsoft Azure + History of Microsoft + Bill Gates + Satya Nadella) — the kind of mixed-relevance dump an analyst research tool would return. The same corpus is reused across sections so the savings numbers are directly comparable, and the analyst question stays fixed: *how big is Microsoft Intelligent Cloud / Azure and what has its growth been?*

In [3]:
from _demo_utils import fetch_corpus, fetch_wikipedia, compresr_diff_html

from langchain_core.tools import tool

CORPUS_TITLES = ['Microsoft', 'Microsoft Azure', 'History of Microsoft', 'Bill Gates', 'Satya Nadella']

@tool
def company_research(query: str) -> str:
    """Fetch a stitched research corpus on a public company."""
    return fetch_corpus(CORPUS_TITLES)

COMPANY = 'Microsoft'
QUERY = 'How large is Microsoft Intelligent Cloud / Azure and what has its growth been?'
raw_article = company_research.invoke({'query': COMPANY})
print(f'Research corpus on {COMPANY}: {len(raw_article):,} chars (~{len(raw_article) // 4:,} tokens)')

Research corpus on Microsoft: 218,717 chars (~54,679 tokens)


## Shared helper — `$` savings table

Same monthly-cost calculator used in all groups so the dollar numbers are comparable. Assumes 1,000 calls/day.

In [4]:
CALLS_PER_DAY = 1_000
DAYS_PER_MONTH = 30

def monthly_cost(tokens: int, price_per_1m: float) -> float:
    return tokens * CALLS_PER_DAY * DAYS_PER_MONTH * price_per_1m / 1_000_000

def print_savings(raw_tokens: int, cmp_tokens: int) -> None:
    prices = {'gpt-5-mini': 0.25, 'gpt-5': 1.25, 'gpt-5.4': 5.00}
    print(f"{'Model':<14}{'Raw $/mo':>14}{'Compresr $/mo':>17}{'Saved $/mo':>14}")
    for name, price in prices.items():
        r = monthly_cost(raw_tokens, price)
        c = monthly_cost(cmp_tokens, price)
        print(f'{name:<14}{r:>14,.2f}{c:>17,.2f}{r - c:>14,.2f}')

## A. Tool wrapper — `wrap_tool_with_compression`

**What it does:** wraps any `StructuredTool` so the output is compressed transparently before being returned.

**When to use it:** your agent is already built and you don't want to touch its construction site. Wrap the tool, swap it in, done. Zero middleware, zero retriever rewiring.


In [5]:
from compresr.integrations.langchain import wrap_tool_with_compression
from openai import OpenAI
from IPython.display import display

smart_lookup = wrap_tool_with_compression(
    company_research,
    api_key=os.environ['COMPRESR_API_KEY'],
    compression_model='latte_v2',
    query_arg='query',
    target_compression_ratio=0.5,
    min_tokens=100,
)

raw_out = company_research.invoke({'query': COMPANY})
cmp_out = smart_lookup.invoke({'query': COMPANY})

oai = OpenAI()
SYS = 'You are a precise equity analyst. Answer only from the provided corpus in 2-3 sentences.'

raw_resp = oai.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'system', 'content': SYS},
              {'role': 'user', 'content': f'Corpus:\n{raw_out}\n\nQuestion: {QUERY}'}],
    temperature=0,
)
cmp_resp = oai.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'system', 'content': SYS},
              {'role': 'user', 'content': f'Corpus:\n{cmp_out}\n\nQuestion: {QUERY}'}],
    temperature=0,
)

raw_in = raw_resp.usage.prompt_tokens
cmp_in = cmp_resp.usage.prompt_tokens
saved_pct = (1 - cmp_in / raw_in) * 100

print(f'Without wrapper: {raw_in:>7,} prompt tokens')
print(f'With wrapper:    {cmp_in:>7,} prompt tokens  ({saved_pct:.1f}% smaller)')
print(f'$ saved / 1k requests at gpt-4o-mini ($0.15/M input): ${(raw_in - cmp_in) * 0.15 / 1000:.3f}')
print()
print('--- Answer WITHOUT compression ---')
print(raw_resp.choices[0].message.content.strip())
print()
print('--- Answer WITH compression ---')
print(cmp_resp.choices[0].message.content.strip())
print()
print_savings(raw_in, cmp_in)
print()
print('Word-level diff (showing first ~20k chars):')
display(compresr_diff_html(raw_out, cmp_out))

/Users/oussama/anaconda3/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Without wrapper:  44,864 prompt tokens
With wrapper:     23,602 prompt tokens  (47.4% smaller)
$ saved / 1k requests at gpt-4o-mini ($0.15/M input): $3.189

--- Answer WITHOUT compression ---
Microsoft Azure, part of the Intelligent Cloud segment, has grown significantly, surpassing $75 billion in annual revenue by the end of fiscal year 2025. The revenue from Cloud Services increased from $16.6 billion in June 2013 to $20.3 billion in June 2014, reflecting a strong upward trajectory in demand for cloud computing solutions.

--- Answer WITH compression ---
Microsoft Azure, part of the Intelligent Cloud segment, surpassed $75 billion in annual revenue by the end of fiscal year 2025 and operates over 400 datacenters across 70 regions. The revenue from Cloud Services grew significantly, from $16.6 billion in June 2011 to $20.3 billion by June 2013, reflecting a strong growth trajectory in Microsoft's cloud computing business.

Model               Raw $/mo    Compresr $/mo    Saved $/mo
gp

## B. Middleware — for `langchain.agents.create_agent`

Three middleware layers cover the three places an agent's prompt blows up:

- **Per-tool**: every tool output, as it enters state (`CompresrToolMiddleware`).
- **Steady-state**: old history rolled into a summary (`CompresrSummarizationMiddleware`).
- **Last-mile**: hard ceiling right before the model is called (`CompresrPromptMiddleware`).

They compose — use one, two, or all three depending on your workload.

### B.1 `CompresrToolMiddleware` — per-tool output compression

**When to use it:** any agent built with `create_agent` where tools return long content (retrievers, web fetchers, API calls). Sits between the tool node and agent state via `wrap_tool_call`. Per-tool allow/ignore lists let you target only the noisy ones.

In [6]:
from langchain_core.messages import HumanMessage, ToolMessage
from compresr.integrations.langchain import CompresrToolMiddleware

tool_mw = CompresrToolMiddleware(
    api_key=os.environ['COMPRESR_API_KEY'],
    allow_tools={'wiki_lookup'},
    compression_model='latte_v2',
    query_arg='query',
    target_compression_ratio=0.5,
)

# Simulate the inner tool returning the raw article. wrap_tool_call sits
# between the tool node and agent state.
def inner_handler(_req):
    return ToolMessage(content=raw_article, tool_call_id='t1', name='wiki_lookup')

class _Req:  # ToolCallRequest stand-in
    pass

req = _Req()
req.tool_call = {'id': 't1', 'name': 'wiki_lookup', 'args': {'query': QUERY}}
req.messages = [HumanMessage(content=QUERY)]

compressed_msg = tool_mw.wrap_tool_call(req, inner_handler)

without = len(raw_article) // 4
with_mw = len(compressed_msg.content) // 4
print(f'Without middleware: {without:>7,} tokens (raw tool output)')
print(f'With middleware:    {with_mw:>7,} tokens   ({(1 - with_mw / without) * 100:.1f}% smaller)')
print()
print('Drop into: create_agent(model=model, tools=[wiki_lookup], middleware=[tool_mw])')

Without middleware:  54,679 tokens (raw tool output)
With middleware:     27,973 tokens   (48.8% smaller)

Drop into: create_agent(model=model, tools=[wiki_lookup], middleware=[tool_mw])


### B.2 `CompresrSummarizationMiddleware` — bounded history

**When to use it:** long-running conversations that accumulate tool outputs and assistant turns. When the message-state crosses `max_tokens_before_summary`, old messages collapse into one summary message and the recent tail stays untouched. Unlike LangChain's built-in `SummarizationMiddleware`, this uses token-level compression (no LLM call) — faster, cheaper, KV-cache friendly.

In [7]:
from langchain_core.messages import HumanMessage, ToolMessage
from compresr.integrations.langchain import CompresrSummarizationMiddleware

summary_mw = CompresrSummarizationMiddleware(
    api_key=os.environ['COMPRESR_API_KEY'],
    compression_model='latte_v2',
    max_tokens_before_summary=4_000,
    messages_to_keep=10,
)

# Build a synthetic conversation over the threshold.
long_tool_out = raw_article[:8_000]
msgs = []
for i in range(8):
    msgs.append(HumanMessage(content=f'Question {i}: tell me more.'))
    msgs.append(ToolMessage(content=long_tool_out, tool_call_id=f't{i}', name='wiki'))

before_tokens = sum(len(m.content) for m in msgs) // 4
out = summary_mw.before_model({'messages': msgs}, runtime=None)
new_msgs = out['messages'][1:]   # index 0 is the RemoveMessage marker
after_tokens = sum(len(m.content) for m in new_msgs) // 4

print(f'Without middleware: {len(msgs):>2} messages, {before_tokens:>7,} tokens (all retained)')
print(f'With middleware:    {len(new_msgs):>2} messages, {after_tokens:>7,} tokens   ({(1 - after_tokens / before_tokens) * 100:.1f}% smaller — 1 summary + recent tail)')
print()
print(f'Summary head: {new_msgs[0].content[:200]}...')

Without middleware: 16 messages,  16,050 tokens (all retained)
With middleware:    11 messages,  13,146 tokens   (18.1% smaller — 1 summary + recent tail)

Summary head: [Earlier conversation summary]

human: Question 0: tell me more.
tool:wiki: Microsoft

Microsoft Corporation is an American multinational technology company headquartered in Redmond, Washington. The c...


## C. Retriever — `CompresrExtractor`

**What it does:** `BaseDocumentCompressor` that compresses retrieved documents in a single batch call.

**When to use it:** RAG pipelines using `ContextualCompressionRetriever`. Drop-in for `LLMChainExtractor` — same shape, but no LLM call per document, just one fast batch compression for the whole list.

In [8]:
from langchain_core.documents import Document
from compresr.integrations.langchain import CompresrExtractor

topics = ['Microsoft', 'Microsoft Azure', 'Amazon Web Services', 'Google Cloud Platform', 'Oracle Cloud']
docs = [Document(page_content=fetch_wikipedia(t), metadata={'title': t}) for t in topics]

extractor = CompresrExtractor(
    api_key=os.environ['COMPRESR_API_KEY'],
    compression_model='latte_v2',
    target_compression_ratio=0.5,
    min_tokens=100,
)
analyst_query = 'Compare the size and growth of the major public cloud providers (Azure, AWS, GCP, Oracle).'
compressed_docs = extractor.compress_documents(docs, query=analyst_query)

without_total = sum(len(d.page_content) for d in docs) // 4
with_total = sum(len(d.page_content) for d in compressed_docs) // 4

print('Per document:')
for orig, cmp in zip(docs, compressed_docs):
    raw_t = len(orig.page_content) // 4
    cmp_t = len(cmp.page_content) // 4
    saved = (1 - cmp_t / raw_t) * 100
    title = orig.metadata['title']
    print(f'  {title:<40}{raw_t:>7,} → {cmp_t:>6,} tokens ({saved:.1f}% smaller)')
print()
print(f'Without extractor: {without_total:>7,} tokens total')
print(f'With extractor:    {with_total:>7,} tokens total ({(1 - with_total / without_total) * 100:.1f}% smaller overall)')

Per document:
  Microsoft                                16,424 →  8,409 tokens (48.8% smaller)
  Microsoft Azure                           6,818 →  3,613 tokens (47.0% smaller)
  Amazon Web Services                       7,719 →  3,981 tokens (48.4% smaller)
  Google Cloud Platform                     3,221 →  1,667 tokens (48.2% smaller)
  Oracle Cloud                              2,465 →  1,289 tokens (47.7% smaller)

Without extractor:  36,648 tokens total
With extractor:     18,961 tokens total (48.3% smaller overall)


## How they compose

Plug the three middlewares into a single `create_agent` call:

```python
from langchain.agents import create_agent
from compresr.integrations.langchain import (
    CompresrToolMiddleware,
    CompresrSummarizationMiddleware,
    CompresrPromptMiddleware,
)

agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        CompresrToolMiddleware(api_key=KEY, query_arg='query'),          # per-tool
        CompresrSummarizationMiddleware(api_key=KEY, max_tokens_before_summary=4_000),
        CompresrPromptMiddleware(api_key=KEY, max_tokens=8_000),         # last-mile cap
    ],
)
```

For RAG pipelines, drop `CompresrExtractor` into a `ContextualCompressionRetriever`.